What should the notebook do?
1. Connect to Google Drive.
2. Locate all 301 ovarian ultrasound images.
3. Verify that each image has a corresponding BiomedParse pseudo-label.
4. Load splits_final.json from the previous nnU-Net experiment.
5. Identify the 240 training cases and 61 validation cases from fold 0.
6. Select 10 pilot images from the training set.
7. Save image_manifest.csv for subsequent experiments.

In [ ]:
from google.colab import drive
from pathlib import Path

# Mount Google Drive
drive.mount("/content/drive")

# Existing BiomedParse / nnU-Net dataset
DATASET_ROOT = Path(
    "/content/drive/MyDrive/biomedparse_project/"
    "nnunet_raw/Dataset001_Follicles"
)

IMAGES_DIR = DATASET_ROOT / "imagesTr"
LABELS_DIR = DATASET_ROOT / "labelsTr"

print("Dataset exists:", DATASET_ROOT.exists())
print("Images directory exists:", IMAGES_DIR.exists())
print("Labels directory exists:", LABELS_DIR.exists())

Mount Google Drive and define the paths to the ultrasound images and pseudo-labels

In [ ]:
# Inspect existing dataset files without modifying them

image_files = sorted(
    p for p in IMAGES_DIR.iterdir() if p.is_file()
)
label_files = sorted(
    p for p in LABELS_DIR.iterdir() if p.is_file()
)

print("Number of image files:", len(image_files))
print("Number of label files:", len(label_files))

print("\nExample image files:")
for path in image_files[:5]:
    print(" ", path.name)

print("\nExample label files:")
for path in label_files[:5]:
    print(" ", path.name)

Count the available images and pseudo-labels and inspect sample filenames

In [ ]:
from collections import Counter

def image_case_id(path):
    if not path.stem.endswith("_0000"):
        raise ValueError(f"Unexpected image filename: {path.name}")
    return path.stem[:-5]

def label_case_id(path):
    return path.stem

image_ids = [image_case_id(p) for p in image_files]
label_ids = [label_case_id(p) for p in label_files]

print("Unique image IDs:", len(set(image_ids)))
print("Unique label IDs:", len(set(label_ids)))

print("Images without labels:", sorted(set(image_ids) - set(label_ids)))
print("Labels without images:", sorted(set(label_ids) - set(image_ids)))

print("\nCases by category:")
for category, count in sorted(
    Counter(case_id.split("_")[0] for case_id in image_ids).items()
):
    print(f"{category}: {count}")

assert len(image_ids) == 301
assert len(label_ids) == 301
assert len(set(image_ids)) == 301
assert len(set(label_ids)) == 301
assert set(image_ids) == set(label_ids)

print("\nSUCCESS: All 301 images have matching pseudo-labels.")

Verify that each ultrasound image has a matching pseudo-label and check the category distribution

In [ ]:
from pathlib import Path

SEARCH_ROOT = Path(
    "/content/drive/MyDrive/biomedparse_project"
)

split_files = sorted(
    SEARCH_ROOT.rglob("splits_final.json")
)

print("Found split files:", len(split_files))

for path in split_files:
    print(path)

Locate the existing nnU-Net split configuration for reproducible train-validation partitioning

In [ ]:
import json
from collections import Counter

# Load the original nnU-Net split configuration
SPLIT_PATH = split_files[0]

with open(SPLIT_PATH, "r", encoding="utf-8") as f:
    splits = json.load(f)

# Select fold 0
fold_0 = splits[0]

train_ids = fold_0["train"]
val_ids = fold_0["val"]

print("Training cases:", len(train_ids))
print("Validation cases:", len(val_ids))

# Verify that the split matches the available dataset
assert len(train_ids) == 240
assert len(val_ids) == 61

assert len(set(train_ids)) == len(train_ids)
assert len(set(val_ids)) == len(val_ids)

assert set(train_ids).isdisjoint(set(val_ids))
assert set(train_ids) | set(val_ids) == set(image_ids)

print("\nSUCCESS: Fold 0 matches all 301 dataset cases.")

# Inspect category distribution
def category_distribution(case_ids):
    return dict(sorted(Counter(
        case_id.split("_")[0] for case_id in case_ids
    ).items()))

print("\nTraining category distribution:")
print(category_distribution(train_ids))

print("\nValidation category distribution:")
print(category_distribution(val_ids))

Load and validate the original nnU-Net fold 0 train-validation split

In [ ]:
import random

SEED = 42
rng = random.Random(SEED)

pilot_distribution = {
    "dominant": 5,
    "normal": 2,
    "pco": 3
}

pilot_ids = []

for category, n_samples in pilot_distribution.items():
    category_ids = sorted(
        case_id for case_id in train_ids
        if case_id.startswith(category + "_")
    )

    selected = rng.sample(category_ids, n_samples)
    pilot_ids.extend(selected)

pilot_ids = sorted(pilot_ids)

assert len(pilot_ids) == 10
assert len(set(pilot_ids)) == 10
assert set(pilot_ids).issubset(set(train_ids))
assert set(pilot_ids).isdisjoint(set(val_ids))

print("Selected pilot cases:")
for case_id in pilot_ids:
    print(" ", case_id)

print("\nPilot category distribution:")
print(category_distribution(pilot_ids))

Select 10 reproducible pilot images from the training set using stratified sampling

In [ ]:
import pandas as pd

# Create a separate directory for the new experiment
OUTPUT_DIR = Path(
    "/content/drive/MyDrive/ovarian_ultrasound_vlm"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

train_set = set(train_ids)
val_set = set(val_ids)
pilot_set = set(pilot_ids)

records = []

for case_id in sorted(image_ids):

    image_path = IMAGES_DIR / f"{case_id}_0000.png"
    label_path = LABELS_DIR / f"{case_id}.png"

    assert image_path.exists(), f"Missing image: {image_path}"
    assert label_path.exists(), f"Missing pseudo-label: {label_path}"

    records.append({
        "case_id": case_id,
        "category": case_id.split("_")[0],
        "image_path": str(image_path),
        "pseudo_label_path": str(label_path),
        "split": "train" if case_id in train_set else "validation",
        "is_pilot": case_id in pilot_set
    })

manifest = pd.DataFrame(records)

# Validate manifest
assert len(manifest) == 301
assert manifest["case_id"].is_unique
assert (manifest["split"] == "train").sum() == 240
assert (manifest["split"] == "validation").sum() == 61
assert manifest["is_pilot"].sum() == 10
assert not manifest.loc[
    manifest["is_pilot"], "split"
].eq("validation").any()

# Save manifest
MANIFEST_PATH = OUTPUT_DIR / "image_manifest.csv"
manifest.to_csv(MANIFEST_PATH, index=False)

print("Manifest saved to:", MANIFEST_PATH)
print("\nDataset summary:")
print(manifest.groupby(["split", "category"]).size())

print("\nPilot images:", manifest["is_pilot"].sum())

display(manifest.head())

Create and save a reproducible image manifest for the foundation model experiments